# 01 — Clean Real Production Data

**Goal:** Load 55-row `production_final.csv`, parse text durations to seconds, compute `Production_End_Time` from cumulative durations, compute per-batch `Avg_OEE`, and export a clean seed.

**Input:** `production_final.csv` (55 rows, 13 cols)  
**Output:** `production_clean.csv` (55 rows, 15 cols)

## Step 1 — Setup

Import libraries. Define paths relative to project root.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ROOT     = Path.cwd()
DATA_DIR = ROOT / "data"
INPUT    = DATA_DIR / "production_final.csv"
OUTPUT   = DATA_DIR / "production_clean.csv"

print(f"Input:  {INPUT}   (exists: {INPUT.exists()})")
print(f"Output: {OUTPUT}")

Input:  E:\Projects\iot-aiml-project\ml_model\data\production_final.csv   (exists: True)
Output: E:\Projects\iot-aiml-project\ml_model\data\production_clean.csv


## Step 2 — Load & Inspect

13 columns. Key facts:
- `Avaliability` has a typo (missing `l`)
- `Availability`, `Performance`, `Quality`, `OEE` are **integers** — keep as int
- 4 text-format duration columns: `Live_Planned_Production_Duration`, `Live_Production_Duration`, `Production_Delay`, `DownTime`
- `Prod_Speed`: discrete levels 0/20/60/100
- No timestamp columns — `Production_End_Time` will be computed from durations

In [2]:
df = pd.read_csv(INPUT)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} cols")
print(f"\nColumns and dtypes:")
for col in df.columns:
    print(f"  {col:45s} {str(df[col].dtype):8s}  unique={df[col].nunique():3d}")
print(f"\nNulls:\n{df.isnull().sum()}")
df.head(3)


Shape: 55 rows, 13 cols

Columns and dtypes:
  OP_Data_SLNo                                  int64     unique= 55
  Batch_PartNo                                  object    unique=  6
  Part_SLNo                                     int64     unique= 16
  Part_No                                       object    unique= 55
  Live_Planned_Production_Duration              object    unique= 21
  Live_Production_Duration                      object    unique= 48
  Production_Delay                              object    unique= 46
  Avaliability                                  int64     unique= 27
  Performance                                   int64     unique= 33
  Quality                                       int64     unique= 14
  OEE                                           int64     unique= 39
  DownTime                                      object    unique= 10
  Prod_Speed                                    int64     unique=  4

Nulls:
OP_Data_SLNo                        0
Batch_PartNo

,OP_Data_SLNo,Batch_PartNo,Part_SLNo,Part_No,Live_Planned_Production_Duration,Live_Production_Duration,Production_Delay,Avaliability,Performance,Quality,OEE,DownTime,Prod_Speed
0,1,WM-23-A-1,1,WM-23-A-1-1,0d 0h 0m 30s,0d 0h 1m 45s,0d 0h 1m 15s,100,29,100,29,0h 0m 0s,20
1,2,WM-23-A-1,2,WM-23-A-1-2,0d 0h 1m 0s,0d 0h 2m 16s,0d 0h 1m 16s,100,44,100,44,0h 0m 0s,60
2,3,WM-23-A-1,3,WM-23-A-1-3,0d 0h 1m 30s,0d 0h 4m 24s,0d 0h 2m 54s,64,34,100,22,0h 0m 32s,60


## Step 3 — Parse Durations (Text → Seconds)

4 columns have text format `"0d 0h 1m 45s"`. Regex extracts each component and sums to total seconds.

Columns parsed:
- `Live_Planned_Production_Duration` → `Planned_Prod_Duration_sec`  
- `Live_Production_Duration` → `Actual_Prod_Duration_sec`  
- `Production_Delay` → `Prod_Delay_sec`  
- `DownTime` → `DownTime_sec`

In [3]:
def parse_to_sec(text: str) -> int:
    d = re.search(r'(\d+)d', str(text))
    h = re.search(r'(\d+)h', str(text))
    m = re.search(r'(\d+)m', str(text))
    s = re.search(r'(\d+)s', str(text))
    return (int(d.group(1)) if d else 0) * 86400 \
         + (int(h.group(1)) if h else 0) * 3600  \
         + (int(m.group(1)) if m else 0) * 60    \
         + (int(s.group(1)) if s else 0)

df["Planned_Prod_Duration_sec"] = df["Live_Planned_Production_Duration"].apply(parse_to_sec)
df["Actual_Prod_Duration_sec"]  = df["Live_Production_Duration"].apply(parse_to_sec)
df["Prod_Delay_sec"]            = df["Production_Delay"].apply(parse_to_sec)
df["DownTime_sec"]              = df["DownTime"].apply(parse_to_sec)

df[["Live_Planned_Production_Duration", "Planned_Prod_Duration_sec",
    "DownTime", "DownTime_sec"]].head(3)


,Live_Planned_Production_Duration,Planned_Prod_Duration_sec,DownTime,DownTime_sec
0,0d 0h 0m 30s,30,0h 0m 0s,0
1,0d 0h 1m 0s,60,0h 0m 0s,0
2,0d 0h 1m 30s,90,0h 0m 32s,32


## Step 4 — Compute Production_End_Time

The raw data has no timestamps. We compute `Production_End_Time` from:
- **Shift start**: `2026-01-01 06:00:00`  
- **Elapsed time**: cumulative sum of `(Actual_Prod_Duration_sec + DownTime_sec)` per row ordered by `OP_Data_SLNo`

In [4]:
SHIFT_START = pd.Timestamp("2026-01-01 06:00:00")

elapsed_sec = (df["Actual_Prod_Duration_sec"] + df["DownTime_sec"]).cumsum()
df["Production_End_Time"] = SHIFT_START + pd.to_timedelta(elapsed_sec, unit="s")

print(f"First:  {df['Production_End_Time'].iloc[0]}")
print(f"Last:   {df['Production_End_Time'].iloc[-1]}")
print(f"Total:  {elapsed_sec.iloc[-1]} sec = {elapsed_sec.iloc[-1]/3600:.1f} hours")


First:  2026-01-01 06:01:45
Last:   2026-01-01 12:42:32
Total:  24152 sec = 6.7 hours


## Step 5 — Fix Typo + Compute OEE_Delta & Avg_OEE

Three changes:

1. **Fix column name**: `Avaliability` → `Availability`  
2. **`OEE_Delta`** = `Current_OEE` − `Previous_OEE` (diff). Positive = improving, negative = degrading.  
3. **`Avg_OEE`** = **per-batch mean** of OEE. All rows in the same `Batch_PartNo` get the same value.

In [5]:
# Fix typo
df.rename(columns={"Avaliability": "Availability"}, inplace=True)

# OEE_Delta = current - previous (int)
df["OEE_Delta"] = df["OEE"].diff().fillna(0).astype(int)

# Avg_OEE = per-batch mean (float — batch average of OEE ints)
df["Avg_OEE"] = df.groupby("Batch_PartNo")["OEE"].transform("mean").round(2)

print(f"OEE_Delta range: {df['OEE_Delta'].min()} to {df['OEE_Delta'].max()}")
print(f"\nPer-batch Avg_OEE:")
print(df.groupby("Batch_PartNo")["Avg_OEE"].first().to_string())


OEE_Delta range: -44 to 40

Per-batch Avg_OEE:
Batch_PartNo
Ra_21_A_2    52.20
Ra_21_A_3    62.57
Ra_21_A_4    52.38
Ra_21_A_5    75.75
Ra_21_A_6    30.50
WM-23-A-1    32.89


## Step 6 — Build Clean DataFrame

Select and rename columns. Keep APQ and OEE as **int** to match source data types.

In [6]:
clean = pd.DataFrame({
    "Production_End_Time":         df["Production_End_Time"],
    "Batch_PartNo":                df["Batch_PartNo"],
    "Part_No":                     df["Part_No"],
    "Part_SLNo":                   df["Part_SLNo"],
    "Planned_Prod_Duration_sec":   df["Planned_Prod_Duration_sec"],
    "Actual_Prod_Duration_sec":    df["Actual_Prod_Duration_sec"],
    "Prod_Delay_sec":              df["Prod_Delay_sec"],
    "Availability":                df["Availability"].astype(int),
    "Performance":                 df["Performance"].astype(int),
    "Quality":                     df["Quality"].astype(int),
    "Current_OEE":                 df["OEE"].astype(int),
    "DownTime_sec":                df["DownTime_sec"],
    "Current_Speed_pct":           df["Prod_Speed"].astype(int),
    "OEE_Delta":                   df["OEE_Delta"],
    "Avg_OEE":                     df["Avg_OEE"],
})

print(f"Clean shape: {clean.shape}")
print(f"Columns: {list(clean.columns)}")
print(f"\nDtypes:")
print(clean.dtypes)
clean.head(3)


Clean shape: (55, 15)
Columns: ['Production_End_Time', 'Batch_PartNo', 'Part_No', 'Part_SLNo', 'Planned_Prod_Duration_sec', 'Actual_Prod_Duration_sec', 'Prod_Delay_sec', 'Availability', 'Performance', 'Quality', 'Current_OEE', 'DownTime_sec', 'Current_Speed_pct', 'OEE_Delta', 'Avg_OEE']

Dtypes:
Production_End_Time          datetime64[ns]
Batch_PartNo                         object
Part_No                              object
Part_SLNo                             int64
Planned_Prod_Duration_sec             int64
Actual_Prod_Duration_sec              int64
Prod_Delay_sec                        int64
Availability                          int64
Performance                           int64
Quality                               int64
Current_OEE                           int64
DownTime_sec                          int64
Current_Speed_pct                     int64
OEE_Delta                             int64
Avg_OEE                             float64
dtype: object


,Production_End_Time,Batch_PartNo,Part_No,Part_SLNo,Planned_Prod_Duration_sec,Actual_Prod_Duration_sec,Prod_Delay_sec,Availability,Performance,Quality,Current_OEE,DownTime_sec,Current_Speed_pct,OEE_Delta,Avg_OEE
0,2026-01-01 06:01:45,WM-23-A-1,WM-23-A-1-1,1,30,105,75,100,29,100,29,0,20,0,32.89
1,2026-01-01 06:04:01,WM-23-A-1,WM-23-A-1-2,2,60,136,76,100,44,100,44,0,60,15,32.89
2,2026-01-01 06:08:57,WM-23-A-1,WM-23-A-1-3,3,90,264,174,64,34,100,22,32,60,-22,32.89


## Step 7 — Summary & Export

Final sanity check before saving to disk.

In [7]:
print("=== Summary ===")
print(clean.describe())
print(f"\n=== Per-batch Avg_OEE ===")
for name, grp in clean.groupby("Batch_PartNo"):
    print(f"  {name:15s} parts={len(grp):2d}  Avg_OEE={grp['Avg_OEE'].iloc[0]:.1f}  OEE_range={grp['Current_OEE'].min()}-{grp['Current_OEE'].max()}")

print(f"\n=== Dtypes ===")
print(clean.dtypes)
print(f"\n=== Nulls ===")
print(clean.isnull().sum())

clean.to_csv(OUTPUT, index=False)
print(f"\nExported: {OUTPUT}")
print(f"  {len(clean)} rows x {len(clean.columns)} cols")


=== Summary ===
                 Production_End_Time  Part_SLNo  Planned_Prod_Duration_sec  \
count                             55  55.000000                  55.000000   
mean   2026-01-01 09:04:39.145454848   5.727273                 292.363636   
min              2026-01-01 06:01:45   1.000000                  30.000000   
25%       2026-01-01 07:13:15.500000   3.000000                 120.000000   
50%              2026-01-01 08:32:06   5.000000                 240.000000   
75%       2026-01-01 11:02:06.500000   8.000000                 360.000000   
max              2026-01-01 12:42:32  16.000000                 960.000000   
std                              NaN   3.753786                 232.035786   

       Actual_Prod_Duration_sec  Prod_Delay_sec  Availability  Performance  \
count                 55.000000       55.000000     55.000000    55.000000   
mean                 375.618182       92.818182     80.418182    72.018182   
min                   94.000000        3.000000


Exported: E:\Projects\iot-aiml-project\ml_model\data\production_clean.csv
  55 rows x 15 cols


# 01 — Clean Real Production Data

**Goal:** Load 55-row `production_final.csv`, parse text durations to seconds, compute `Production_End_Time` from cumulative durations, compute per-batch `Avg_OEE`, and export a clean seed.

**Input:** `production_final.csv` (55 rows, 13 cols)  
**Output:** `production_clean.csv` (55 rows, 15 cols)

## Step 1 — Setup

Import libraries. Define paths relative to project root.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ROOT     = Path.cwd()
DATA_DIR = ROOT / "data"
INPUT    = DATA_DIR / "production_final.csv"
OUTPUT   = DATA_DIR / "production_clean.csv"

print(f"Input:  {INPUT}   (exists: {INPUT.exists()})")
print(f"Output: {OUTPUT}")

Input:  E:\Projects\iot-aiml-project\ml_model\data\production_final.csv   (exists: True)
Output: E:\Projects\iot-aiml-project\ml_model\data\production_clean.csv


## Step 2 — Load & Inspect

13 columns. Key facts:
- `Avaliability` has a typo (missing `l`)
- `Availability`, `Performance`, `Quality`, `OEE` are **integers** — keep as int
- 4 text-format duration columns: `Live_Planned_Production_Duration`, `Live_Production_Duration`, `Production_Delay`, `DownTime`
- `Prod_Speed`: discrete levels 0/20/60/100
- No timestamp columns — `Production_End_Time` will be computed from durations

In [2]:
df = pd.read_csv(INPUT)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} cols")
print(f"\nColumns and dtypes:")
for col in df.columns:
    print(f"  {col:45s} {str(df[col].dtype):8s}  unique={df[col].nunique():3d}")
print(f"\nNulls:\n{df.isnull().sum()}")
df.head(3)


Shape: 55 rows, 13 cols

Columns and dtypes:
  OP_Data_SLNo                                  int64     unique= 55
  Batch_PartNo                                  object    unique=  6
  Part_SLNo                                     int64     unique= 16
  Part_No                                       object    unique= 55
  Live_Planned_Production_Duration              object    unique= 21
  Live_Production_Duration                      object    unique= 48
  Production_Delay                              object    unique= 46
  Avaliability                                  int64     unique= 27
  Performance                                   int64     unique= 33
  Quality                                       int64     unique= 14
  OEE                                           int64     unique= 39
  DownTime                                      object    unique= 10
  Prod_Speed                                    int64     unique=  4

Nulls:
OP_Data_SLNo                        0
Batch_PartNo

,OP_Data_SLNo,Batch_PartNo,Part_SLNo,Part_No,Live_Planned_Production_Duration,Live_Production_Duration,Production_Delay,Avaliability,Performance,Quality,OEE,DownTime,Prod_Speed
0,1,WM-23-A-1,1,WM-23-A-1-1,0d 0h 0m 30s,0d 0h 1m 45s,0d 0h 1m 15s,100,29,100,29,0h 0m 0s,20
1,2,WM-23-A-1,2,WM-23-A-1-2,0d 0h 1m 0s,0d 0h 2m 16s,0d 0h 1m 16s,100,44,100,44,0h 0m 0s,60
2,3,WM-23-A-1,3,WM-23-A-1-3,0d 0h 1m 30s,0d 0h 4m 24s,0d 0h 2m 54s,64,34,100,22,0h 0m 32s,60


## Step 3 — Parse Durations (Text → Seconds)

4 columns have text format `"0d 0h 1m 45s"`. Regex extracts each component and sums to total seconds.

Columns parsed:
- `Live_Planned_Production_Duration` → `Planned_Prod_Duration_sec`  
- `Live_Production_Duration` → `Actual_Prod_Duration_sec`  
- `Production_Delay` → `Prod_Delay_sec`  
- `DownTime` → `DownTime_sec`

In [3]:
def parse_to_sec(text: str) -> int:
    d = re.search(r'(\d+)d', str(text))
    h = re.search(r'(\d+)h', str(text))
    m = re.search(r'(\d+)m', str(text))
    s = re.search(r'(\d+)s', str(text))
    return (int(d.group(1)) if d else 0) * 86400 \
         + (int(h.group(1)) if h else 0) * 3600  \
         + (int(m.group(1)) if m else 0) * 60    \
         + (int(s.group(1)) if s else 0)

df["Planned_Prod_Duration_sec"] = df["Live_Planned_Production_Duration"].apply(parse_to_sec)
df["Actual_Prod_Duration_sec"]  = df["Live_Production_Duration"].apply(parse_to_sec)
df["Prod_Delay_sec"]            = df["Production_Delay"].apply(parse_to_sec)
df["DownTime_sec"]              = df["DownTime"].apply(parse_to_sec)

df[["Live_Planned_Production_Duration", "Planned_Prod_Duration_sec",
    "DownTime", "DownTime_sec"]].head(3)


,Live_Planned_Production_Duration,Planned_Prod_Duration_sec,DownTime,DownTime_sec
0,0d 0h 0m 30s,30,0h 0m 0s,0
1,0d 0h 1m 0s,60,0h 0m 0s,0
2,0d 0h 1m 30s,90,0h 0m 32s,32


## Step 4 — Compute Production_End_Time

The raw data has no timestamps. We compute `Production_End_Time` from:
- **Shift start**: `2026-01-01 06:00:00`  
- **Elapsed time**: cumulative sum of `(Actual_Prod_Duration_sec + DownTime_sec)` per row ordered by `OP_Data_SLNo`

In [4]:
SHIFT_START = pd.Timestamp("2026-01-01 06:00:00")

elapsed_sec = (df["Actual_Prod_Duration_sec"] + df["DownTime_sec"]).cumsum()
df["Production_End_Time"] = SHIFT_START + pd.to_timedelta(elapsed_sec, unit="s")

print(f"First:  {df['Production_End_Time'].iloc[0]}")
print(f"Last:   {df['Production_End_Time'].iloc[-1]}")
print(f"Total:  {elapsed_sec.iloc[-1]} sec = {elapsed_sec.iloc[-1]/3600:.1f} hours")


First:  2026-01-01 06:01:45
Last:   2026-01-01 12:42:32
Total:  24152 sec = 6.7 hours


## Step 5 — Fix Typo + Compute OEE_Delta & Avg_OEE

Three changes:

1. **Fix column name**: `Avaliability` → `Availability`  
2. **`OEE_Delta`** = `Current_OEE` − `Previous_OEE` (diff). Positive = improving, negative = degrading.  
3. **`Avg_OEE`** = **per-batch mean** of OEE. All rows in the same `Batch_PartNo` get the same value.

In [5]:
# Fix typo
df.rename(columns={"Avaliability": "Availability"}, inplace=True)

# OEE_Delta = current - previous (int)
df["OEE_Delta"] = df["OEE"].diff().fillna(0).astype(int)

# Avg_OEE = per-batch mean (float — batch average of OEE ints)
df["Avg_OEE"] = df.groupby("Batch_PartNo")["OEE"].transform("mean").round(2)

print(f"OEE_Delta range: {df['OEE_Delta'].min()} to {df['OEE_Delta'].max()}")
print(f"\nPer-batch Avg_OEE:")
print(df.groupby("Batch_PartNo")["Avg_OEE"].first().to_string())


OEE_Delta range: -44 to 40

Per-batch Avg_OEE:
Batch_PartNo
Ra_21_A_2    52.20
Ra_21_A_3    62.57
Ra_21_A_4    52.38
Ra_21_A_5    75.75
Ra_21_A_6    30.50
WM-23-A-1    32.89


## Step 6 — Build Clean DataFrame

Select and rename columns. Keep APQ and OEE as **int** to match source data types.

In [6]:
clean = pd.DataFrame({
    "Production_End_Time":         df["Production_End_Time"],
    "Batch_PartNo":                df["Batch_PartNo"],
    "Part_No":                     df["Part_No"],
    "Part_SLNo":                   df["Part_SLNo"],
    "Planned_Prod_Duration_sec":   df["Planned_Prod_Duration_sec"],
    "Actual_Prod_Duration_sec":    df["Actual_Prod_Duration_sec"],
    "Prod_Delay_sec":              df["Prod_Delay_sec"],
    "Availability":                df["Availability"].astype(int),
    "Performance":                 df["Performance"].astype(int),
    "Quality":                     df["Quality"].astype(int),
    "Current_OEE":                 df["OEE"].astype(int),
    "DownTime_sec":                df["DownTime_sec"],
    "Current_Speed_pct":           df["Prod_Speed"].astype(int),
    "OEE_Delta":                   df["OEE_Delta"],
    "Avg_OEE":                     df["Avg_OEE"],
})

print(f"Clean shape: {clean.shape}")
print(f"Columns: {list(clean.columns)}")
print(f"\nDtypes:")
print(clean.dtypes)
clean.head(3)


Clean shape: (55, 15)
Columns: ['Production_End_Time', 'Batch_PartNo', 'Part_No', 'Part_SLNo', 'Planned_Prod_Duration_sec', 'Actual_Prod_Duration_sec', 'Prod_Delay_sec', 'Availability', 'Performance', 'Quality', 'Current_OEE', 'DownTime_sec', 'Current_Speed_pct', 'OEE_Delta', 'Avg_OEE']

Dtypes:
Production_End_Time          datetime64[ns]
Batch_PartNo                         object
Part_No                              object
Part_SLNo                             int64
Planned_Prod_Duration_sec             int64
Actual_Prod_Duration_sec              int64
Prod_Delay_sec                        int64
Availability                          int64
Performance                           int64
Quality                               int64
Current_OEE                           int64
DownTime_sec                          int64
Current_Speed_pct                     int64
OEE_Delta                             int64
Avg_OEE                             float64
dtype: object


,Production_End_Time,Batch_PartNo,Part_No,Part_SLNo,Planned_Prod_Duration_sec,Actual_Prod_Duration_sec,Prod_Delay_sec,Availability,Performance,Quality,Current_OEE,DownTime_sec,Current_Speed_pct,OEE_Delta,Avg_OEE
0,2026-01-01 06:01:45,WM-23-A-1,WM-23-A-1-1,1,30,105,75,100,29,100,29,0,20,0,32.89
1,2026-01-01 06:04:01,WM-23-A-1,WM-23-A-1-2,2,60,136,76,100,44,100,44,0,60,15,32.89
2,2026-01-01 06:08:57,WM-23-A-1,WM-23-A-1-3,3,90,264,174,64,34,100,22,32,60,-22,32.89


## Step 7 — Summary & Export

Final sanity check before saving to disk.

In [7]:
print("=== Summary ===")
print(clean.describe())
print(f"\n=== Per-batch Avg_OEE ===")
for name, grp in clean.groupby("Batch_PartNo"):
    print(f"  {name:15s} parts={len(grp):2d}  Avg_OEE={grp['Avg_OEE'].iloc[0]:.1f}  OEE_range={grp['Current_OEE'].min()}-{grp['Current_OEE'].max()}")

print(f"\n=== Dtypes ===")
print(clean.dtypes)
print(f"\n=== Nulls ===")
print(clean.isnull().sum())

clean.to_csv(OUTPUT, index=False)
print(f"\nExported: {OUTPUT}")
print(f"  {len(clean)} rows x {len(clean.columns)} cols")


=== Summary ===
                 Production_End_Time  Part_SLNo  Planned_Prod_Duration_sec  \
count                             55  55.000000                  55.000000   
mean   2026-01-01 09:04:39.145454848   5.727273                 292.363636   
min              2026-01-01 06:01:45   1.000000                  30.000000   
25%       2026-01-01 07:13:15.500000   3.000000                 120.000000   
50%              2026-01-01 08:32:06   5.000000                 240.000000   
75%       2026-01-01 11:02:06.500000   8.000000                 360.000000   
max              2026-01-01 12:42:32  16.000000                 960.000000   
std                              NaN   3.753786                 232.035786   

       Actual_Prod_Duration_sec  Prod_Delay_sec  Availability  Performance  \
count                 55.000000       55.000000     55.000000    55.000000   
mean                 375.618182       92.818182     80.418182    72.018182   
min                   94.000000        3.000000


Exported: E:\Projects\iot-aiml-project\ml_model\data\production_clean.csv
  55 rows x 15 cols
